# Fine-tune YOLO cho Window Detection — vUDT Sub-module 4

**Mục tiêu**: So sánh YOLOv8s, YOLOv8m, YOLOv11s, YOLOv11m trên bộ dữ liệu window detection (986/122/122 ảnh train/val/test, 61,812 bbox, single-class).

**Tham số cố định**: `imgsz=1024`, `epochs=100`, `patience=20`, `fliplr=0.5`, `degrees=0.0`.

**Lịch chạy đề xuất** (do giới hạn session Kaggle ~12h và quota GPU tuần):
| Nhóm | imgsz | Thời gian/model ước tính |
|---|---|---|
| YOLOv8s, YOLOv11s | 1024 | ~1-2 giờ |
| YOLOv8m, YOLOv11m | 1024 | ~2-3.5 giờ |

RT-DETR **không** nằm trong notebook này — sẽ làm ở notebook/session riêng sau vì pipeline augment khác YOLO và nặng hơn đáng kể.

**Tập `test` chỉ được đụng tới ở Mục 4 (đánh giá cuối cùng)** — hoàn toàn không xuất hiện trong `dataset.yaml` phần train/val, không được dùng trong vòng lặp training, đảm bảo đúng nguyên tắc blind test.

Notebook có checkpoint/resume (`save_period=10`) để chống mất tiến độ nếu session Kaggle bị ngắt giữa chừng.

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
# Xác nhận tên dataset/slug đã mount
!ls /kaggle/input/

In [ ]:
import os

# ============================================================
# ⚠️ SỬA DÒNG DƯỚI: đường dẫn tới thư mục data_Train sau khi mount
# Ví dụ: "/kaggle/input/window-detection-data/data_Train"
KAGGLE_DATA_ROOT = "/kaggle/input/TEN_DATASET_CUA_BAN/data_Train"  # <-- SỬA DÒNG NÀY
# ============================================================

WORK_DIR = "/kaggle/working"
os.makedirs(WORK_DIR, exist_ok=True)

# Kiểm tra cấu trúc thư mục có đúng như kỳ vọng không
assert os.path.isdir(KAGGLE_DATA_ROOT), f"Không tìm thấy: {KAGGLE_DATA_ROOT} — kiểm tra lại slug/cấu trúc zip ở cell trên"
for sub in ["images/train", "images/val", "images/test", "labels/train", "labels/val", "labels/test"]:
    p = os.path.join(KAGGLE_DATA_ROOT, sub)
    n = len(os.listdir(p)) if os.path.isdir(p) else -1
    print(f"{sub:20s} -> {n} files" if n >= 0 else f"{sub:20s} -> KHÔNG TỒN TẠI, kiểm tra lại đường dẫn")

In [ ]:
# dataset.yaml gốc trỏ tới đường dẫn Windows local (e:/Viettel Solution/dataYolo/data_Train)
# -> phải viết lại với đường dẫn Kaggle, KHÔNG dùng file dataset.yaml gốc trong zip

yaml_content = f"""path: {KAGGLE_DATA_ROOT}
train: images/train
val: images/val
test: images/test

names:
  0: window
"""

yaml_path = os.path.join(WORK_DIR, "dataset.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(yaml_content)

## 2. Cấu hình training

**Lưu ý GPU trên Kaggle**: accelerator có thể là **P100 (1 GPU)** hoặc **T4 x2 (2 GPU)**.
- Nếu 1 GPU: `device=0`, `batch=-1` (auto batch theo VRAM) chạy an toàn.
- Nếu 2 GPU (T4x2): `batch=-1` (auto) không tương thích tốt với multi-GPU — nên đặt `device=[0,1]` và **tự set batch cố định** (ví dụ `batch=16` cho model "s", `batch=8` cho model "m" ở imgsz=1024), script bên dưới có xử lý việc này qua biến `NUM_GPUS`.

In [ ]:
from ultralytics import YOLO
import torch

NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
DEVICE = list(range(NUM_GPUS)) if NUM_GPUS > 1 else (0 if NUM_GPUS == 1 else "cpu")
print("Device:", DEVICE, "| GPU count:", NUM_GPUS)

# ============================================================
# ⚠️ CHỌN model chạy trong session này — chia nhỏ để tránh vượt 12h/session
# Session 1: MODELS_TO_RUN = ["yolov8s", "yolo11s"]   (~1-2h/model)
# Session 2: MODELS_TO_RUN = ["yolov8m", "yolo11m"]   (~2-3.5h/model)
MODELS_TO_RUN = ["yolov8s", "yolo11s"]  # <-- SỬA THEO SESSION
# ============================================================

MODEL_WEIGHTS = {
    "yolov8s": "yolov8s.pt",
    "yolov8m": "yolov8m.pt",
    "yolo11s": "yolo11s.pt",
    "yolo11m": "yolo11m.pt",
}

# batch cố định dùng khi multi-GPU (NUM_GPUS > 1); bỏ qua nếu chạy 1 GPU (dùng auto batch=-1)
FIXED_BATCH_MULTI_GPU = {
    "yolov8s": 16, "yolo11s": 16,
    "yolov8m": 8,  "yolo11m": 8,
}

BASE_TRAIN_ARGS = dict(
    data=yaml_path,
    imgsz=1024,
    epochs=100,
    patience=20,
    fliplr=0.5,
    degrees=0.0,
    device=DEVICE,
    project=f"{WORK_DIR}/runs",
    exist_ok=True,
    plots=True,
    save=True,
    save_period=10,   # lưu checkpoint mỗi 10 epoch để resume nếu session bị ngắt giữa chừng
)

## 3. Training (có resume nếu tìm thấy checkpoint cũ)

In [ ]:
results_summary = {}

for model_name in MODELS_TO_RUN:
    print(f"\n{'='*60}\nTraining {model_name}\n{'='*60}")
    last_ckpt = f"{WORK_DIR}/runs/{model_name}/weights/last.pt"

    train_args = dict(BASE_TRAIN_ARGS)
    if NUM_GPUS > 1:
        train_args["batch"] = FIXED_BATCH_MULTI_GPU[model_name]
    else:
        train_args["batch"] = -1  # auto batch, chỉ an toàn với 1 GPU

    if os.path.exists(last_ckpt):
        print(f"Tìm thấy checkpoint -> resume từ {last_ckpt}")
        model = YOLO(last_ckpt)
        train_result = model.train(resume=True)
    else:
        model = YOLO(MODEL_WEIGHTS[model_name])
        train_result = model.train(name=model_name, **train_args)

    results_summary[model_name] = train_result

## 4. Đánh giá độc lập trên tập TEST (blind test)

Tập `test` **không được dùng ở bất kỳ đâu trong Mục 3** (không train, không validate trong lúc train) — chỉ dùng đúng 1 lần ở đây để chấm điểm cuối cùng, đúng nguyên tắc held-out blind test.

Metrics tính cho từng model:
- **Precision, Recall** — lấy tại ngưỡng confidence Ultralytics chọn tối ưu F1 (từ `model.val()`)
- **AP50, AP75, mAP50:95** — vì dataset single-class nên AP50 = mAP50, AP75 = mAP75
- **Mean IoU (TP)** — IoU trung bình của các detection khớp đúng (matched, IoU ≥ 0.5) — Ultralytics không xuất sẵn số này nên tính thủ công bằng `box_iou` + greedy matching
- **Avg detect time (ms/ảnh)** — thời gian inference trung bình mỗi ảnh, đo trực tiếp qua `model.predict()`

In [ ]:
import numpy as np
from pathlib import Path
from PIL import Image
from ultralytics.utils.metrics import box_iou
import torch

def yolo_to_xyxy(label_path, img_w, img_h):
    """Đọc file nhãn YOLO (.txt) và chuyển sang toạ độ pixel xyxy."""
    boxes = []
    if not os.path.exists(label_path):
        return np.zeros((0, 4), dtype=np.float32)
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            _, xc, yc, w, h = map(float, parts[:5])
            x1 = (xc - w / 2) * img_w
            y1 = (yc - h / 2) * img_h
            x2 = (xc + w / 2) * img_w
            y2 = (yc + h / 2) * img_h
            boxes.append([x1, y1, x2, y2])
    return np.array(boxes, dtype=np.float32)


def eval_mean_iou_and_speed(model, img_dir, lbl_dir, imgsz=1024, conf=0.25, iou_match_thres=0.5):
    """
    Chạy predict trên toàn bộ ảnh test, ghép detection <-> ground-truth bằng greedy
    matching theo IoU (giống cách gán TP trong mAP), trả về mean IoU của các cặp TP
    và thời gian inference trung bình mỗi ảnh (ms).
    """
    img_dir = Path(img_dir)
    lbl_dir = Path(lbl_dir)
    img_files = sorted([p for p in img_dir.glob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])

    matched_ious = []
    infer_times_ms = []

    for img_path in img_files:
        result = model.predict(source=str(img_path), imgsz=imgsz, conf=conf, verbose=False)[0]
        infer_times_ms.append(result.speed["inference"])

        with Image.open(img_path) as im:
            img_w, img_h = im.size

        gt_boxes = yolo_to_xyxy(lbl_dir / (img_path.stem + ".txt"), img_w, img_h)
        pred_boxes = result.boxes.xyxy.cpu().numpy() if len(result.boxes) else np.zeros((0, 4), dtype=np.float32)

        if len(gt_boxes) == 0 or len(pred_boxes) == 0:
            continue

        iou_matrix = box_iou(torch.tensor(pred_boxes), torch.tensor(gt_boxes)).numpy()

        used_pred, used_gt = set(), set()
        pairs = sorted(
            ((iou_matrix[p, g], p, g) for p in range(iou_matrix.shape[0]) for g in range(iou_matrix.shape[1])),
            key=lambda x: x[0], reverse=True,
        )
        for iou_val, p, g in pairs:
            if iou_val < iou_match_thres:
                break
            if p in used_pred or g in used_gt:
                continue
            used_pred.add(p)
            used_gt.add(g)
            matched_ious.append(iou_val)

    mean_iou = float(np.mean(matched_ious)) if matched_ious else 0.0
    avg_time_ms = float(np.mean(infer_times_ms)) if infer_times_ms else 0.0
    return mean_iou, avg_time_ms

In [ ]:
final_metrics = {}

for model_name in MODELS_TO_RUN:
    best_pt = f"{WORK_DIR}/runs/{model_name}/weights/best.pt"
    if not os.path.exists(best_pt):
        print(f"Bỏ qua {model_name}: chưa có best.pt")
        continue

    model = YOLO(best_pt)

    # Precision / Recall / AP50 / AP75 / mAP50:95 - qua validator chuẩn của Ultralytics, split="test"
    val_metrics = model.val(
        data=yaml_path,
        split="test",
        imgsz=1024,
        device=0 if NUM_GPUS >= 1 else "cpu",
        project=f"{WORK_DIR}/runs_test",
        name=model_name,
        plots=True,
    )

    # Mean IoU (TP) + thời gian detect trung bình - tính thủ công vì Ultralytics không xuất sẵn
    test_img_dir = os.path.join(KAGGLE_DATA_ROOT, "images/test")
    test_lbl_dir = os.path.join(KAGGLE_DATA_ROOT, "labels/test")
    mean_iou, avg_time_ms = eval_mean_iou_and_speed(model, test_img_dir, test_lbl_dir, imgsz=1024)

    final_metrics[model_name] = {
        "Precision": val_metrics.box.mp,
        "Recall": val_metrics.box.mr,
        "AP50": val_metrics.box.map50,
        "AP75": val_metrics.box.map75,
        "mAP50:95": val_metrics.box.map,
        "Mean IoU (TP)": mean_iou,
        "Avg detect time (ms/img)": avg_time_ms,
    }

df = pd.DataFrame(final_metrics).T
df

In [ ]:
import pandas as pd
df.to_csv(f"{WORK_DIR}/test_eval_summary.csv")
df